#Cálculo da WER

In [10]:
!pip install jiwer

In [11]:
import jiwer
import re

In [12]:
_PUNCT_CHAR = r"(?!(?<=\w)['’-](?=\w))[^\w\s]"
_PUNCT_SUB = re.compile(f"({_PUNCT_CHAR})")
_IS_PUNCT = re.compile(r"^[^\w\s]$")
PUNCT = "[PUNCT]"


def normalize_text(text):
    text = text.lower()
    text = re.sub(_PUNCT_CHAR, "", text)
    text = re.sub(r"\s+", " ", text)
    return text.strip()


def normalize_separate_punctuation(text):
    text = text.lower()
    text = _PUNCT_SUB.sub(r" \1 ", text)
    text = re.sub(r"\s+", " ", text)
    return text.strip()

In [13]:
def _mask(tokens):
  return [PUNCT if _IS_PUNCT.match(t) else t for t in tokens]


def _align_punctuation(ref, hyp):
  """Alinha as sequências mascaradas e devolve os pares (i, j) de
  pontuações alinhadas entre si."""
  ref_m, hyp_m = _mask(ref), _mask(hyp)
  n, m = len(ref), len(hyp)
  big = n + m + 1

  def step(i, j):
    if ref_m[i] != hyp_m[j]:
      return big
    if ref_m[i] == PUNCT and ref[i] != hyp[j]:
      return 1
    return 0

  dist = [[0] * (m + 1) for _ in range(n + 1)]
  for i in range(1, n + 1):
    dist[i][0] = i * big
  for j in range(1, m + 1):
    dist[0][j] = j * big
  for i in range(1, n + 1):
    for j in range(1, m + 1):
        dist[i][j] = min(
          dist[i - 1][j - 1] + step(i - 1, j - 1),
          dist[i - 1][j] + big,
          dist[i][j - 1] + big,
        )

  pairs, i, j = [], n, m
  while i > 0 and j > 0:
    if dist[i][j] == dist[i - 1][j - 1] + step(i - 1, j - 1):
      if ref_m[i - 1] == PUNCT and hyp_m[j - 1] == PUNCT:
          pairs.append((i - 1, j - 1))
      i, j = i - 1, j - 1
    elif dist[i][j] == dist[i - 1][j] + big:
      i -= 1
    else:
      j -= 1
  return pairs


def punctuation_counts(reference, hypothesis):
  """Espera textos já passados por normalize_separate_punctuation."""
  ref, hyp = reference.split(), hypothesis.split()
  pairs = _align_punctuation(ref, hyp)

  n_ref = sum(1 for t in ref if _IS_PUNCT.match(t))
  n_hyp = sum(1 for t in hyp if _IS_PUNCT.match(t))
  correct = sum(ref[i] == hyp[j] for i, j in pairs)
  return {
    "C": correct,
    "S": len(pairs) - correct,
    "D": n_ref - len(pairs),
    "I": n_hyp - len(pairs),
  }


def _per(c):
  errors = c["S"] + c["D"] + c["I"]
  total = errors + c["C"]
  return errors / total if total else 0.0


def punctuation_error_rate(reference, hypothesis):
  """
  Cálculo da Punctuation Error Rate (PER), com base em https://arxiv.org/abs/2310.02943

  PER = Ip + Dp + Sp / Ip + Dp + Sp + Cp
  Ip = Pontuações Inseridas
  Dp = Pontuações Deletadas
  Sp = Pontuações Substituídas
  Cp = Pontuações Corretas
  """
  reference = normalize_separate_punctuation(reference)
  hypothesis = normalize_separate_punctuation(hypothesis)
  return _per(punctuation_counts(reference, hypothesis))


def corpus_punctuation_error_rate(references, hypotheses):
  """ PER de um corpus todo"""
  total = {"C": 0, "S": 0, "D": 0, "I": 0}
  for r, h in zip(references, hypotheses):
    for k, v in punctuation_counts(r, h).items():
      total[k] += v
  return _per(total)

In [16]:
import csv
import json
import unicodedata
from pathlib import Path

ARQUIVO_GABARITOS = "dicionario (1).json"
ARQUIVO_RESULTADOS = "resultados_transcricao (2).json"


def normalize_filename(filename):
    # Os caminhos nos JSONs podem ter pastas diferentes ou Unicode composto/decomposto.
    return unicodedata.normalize("NFC", Path(filename).name)


with open(ARQUIVO_GABARITOS, encoding="utf-8") as f:
    gabaritos = json.load(f)

with open(ARQUIVO_RESULTADOS, encoding="utf-8") as f:
    resultados = json.load(f)

gabaritos_por_arquivo = {normalize_filename(nome): texto for nome, texto in gabaritos.items()}
linhas_csv = []

for imagem in resultados["imagens"]:
    nome_arquivo = normalize_filename(imagem["arquivo"])
    referencia = gabaritos_por_arquivo.get(nome_arquivo)
    transcricao = imagem["texto_transcrito"]

    linha = {
        "arquivo": nome_arquivo,
        "modelo": resultados.get("modelo", ""),
        "esforco_raciocinio": resultados.get("esforco_raciocinio", ""),
        "custo": imagem.get("custo", ""),
    }

    if referencia is None:
        print(f"\n{nome_arquivo}: gabarito não encontrado")
        linha["status"] = "gabarito não encontrado"
        for metrica in ("wer_sem_pontuacao", "cer_sem_pontuacao", "wer_com_pontuacao", "cer_com_pontuacao", "per"):
            linha[metrica] = ""
        linhas_csv.append(linha)
        continue

    ref_sem_pontuacao = normalize_text(referencia)
    hyp_sem_pontuacao = normalize_text(transcricao)
    ref_com_pontuacao = normalize_separate_punctuation(referencia)
    hyp_com_pontuacao = normalize_separate_punctuation(transcricao)

    metricas = {
        "wer_sem_pontuacao": jiwer.wer(ref_sem_pontuacao, hyp_sem_pontuacao),
        "cer_sem_pontuacao": jiwer.cer(ref_sem_pontuacao, hyp_sem_pontuacao),
        "wer_com_pontuacao": jiwer.wer(ref_com_pontuacao, hyp_com_pontuacao),
        "cer_com_pontuacao": jiwer.cer(ref_com_pontuacao, hyp_com_pontuacao),
        "per": punctuation_error_rate(ref_com_pontuacao, hyp_com_pontuacao),
    }
    linha.update(metricas)
    linha["status"] = "ok"
    linhas_csv.append(linha)

    print(f"\nImagem: {nome_arquivo}")
    print(f"WER (sem pontuação): {metricas['wer_sem_pontuacao']:.4f}")
    print(f"CER (sem pontuação): {metricas['cer_sem_pontuacao']:.4f}")
    print(f"WER (com pontuação): {metricas['wer_com_pontuacao']:.4f}")
    print(f"CER (com pontuação): {metricas['cer_com_pontuacao']:.4f}")
    print(f"PER: {metricas['per']:.4f}")

# O nome do CSV acompanha o arquivo JSON de resultados, preservando cada conjunto.
arquivo_csv = Path(ARQUIVO_RESULTADOS).with_name(
    f"{Path(ARQUIVO_RESULTADOS).stem}_metricas.csv"
)
colunas = [
    "arquivo", "modelo", "esforco_raciocinio", "custo", "status",
    "wer_sem_pontuacao", "cer_sem_pontuacao",
    "wer_com_pontuacao", "cer_com_pontuacao", "per",
]
with open(arquivo_csv, "w", newline="", encoding="utf-8-sig") as f:
    escritor = csv.DictWriter(f, fieldnames=colunas)
    escritor.writeheader()
    escritor.writerows(linhas_csv)

print(f"\nResultados salvos em: {arquivo_csv}")



Imagem: 5.BR FPMSC FUN CAM LDL - Livro de Registro de Leis n° 1, 1892 - 1894.jpg
WER (sem pontuação): 0.1778
CER (sem pontuação): 0.0784
WER (com pontuação): 0.1800
CER (com pontuação): 0.0863
PER: 0.2857

Imagem: 6.BR FPMSC FUN CAM LDL - Livro de Registro de Leis n° 1, 1892 - 1984.jpg
WER (sem pontuação): 0.0781
CER (sem pontuação): 0.0227
WER (com pontuação): 0.0943
CER (com pontuação): 0.0303
PER: 0.2174

Imagem: 7.BR FPMSC FUN CAM LDL - Livro de Registro de Leis n° 1, 1892 - 1984.jpg
WER (sem pontuação): 0.1280
CER (sem pontuação): 0.0412
WER (com pontuação): 0.1210
CER (com pontuação): 0.0458
PER: 0.1282

Imagem: 8.BR FPMSC FUN CAM LDL - Livro de Registro de Leis n° 1, 1892 - 1984.jpg
WER (sem pontuação): 0.1163
CER (sem pontuação): 0.0387
WER (com pontuação): 0.1165
CER (com pontuação): 0.0419
PER: 0.1081

Imagem: 9.BR FPMSC FUN CAM LDL - Livro de Registro de Leis n° 1, 1892 - 1984.jpg
WER (sem pontuação): 0.1608
CER (sem pontuação): 0.0651
WER (com pontuação): 0.1730
CER (com p